# PANDAS MASTERY — CHAPTERS 13–18

```
PANDAS MASTERY
Current Chapters: 13 (Duplicates), 14 (Sorting), 15 (Modifying Data),
                  16 (String Operations), 17 (Categorical Data),
                  18 (Apply/Map/Applymap)
Progress:         18 / 76 chapters
Completed:        Ch1-12
Remaining:        58 chapters
```

Six chapters: finding and handling duplicate records correctly (not just blindly), sorting,
modifying/creating columns the professional way, the string-cleaning toolkit that real-world
messy text data demands, categorical dtype for memory/performance, and the
apply/map/vectorization decision tree.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)


---
# CHAPTER 13 — Duplicates

## Concept
`duplicated()` flags rows that are duplicates of an *earlier* row (by default); `drop_duplicates()`
removes them. The real skill isn't the syntax -- it's knowing **what "duplicate" means for your
specific data**, which is a business question, not a mechanical one.


In [ ]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6],
    "customer_id": [101, 102, 101, 103, 102, 101],
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Gadget", "Gizmo"],
    "amount": [29.99, 49.99, 29.99, 19.99, 49.99, 19.99],
})

print("orders:\n", orders)
print()
print("duplicated() (exact full-row duplicates, default keep='first'):\n", orders.duplicated())
print()
print("duplicated(subset=['customer_id','product']) (same customer+product, ignoring order_id/amount):")
print(orders.duplicated(subset=["customer_id", "product"]))


### Expected Output (verified on pandas 3.0.2)
```
orders:
   order_id  customer_id product  amount
0         1          101  Widget   29.99
1         2          102  Gadget   49.99
2         3          101  Widget   29.99
3         4          103   Gizmo   19.99
4         5          102  Gadget   49.99
5         6          101   Gizmo   19.99

duplicated() (exact full-row duplicates, default keep='first'):
0    False
1    False
2    False
3    False
4    False
5    False
dtype: bool

duplicated(subset=['customer_id','product']) (same customer+product, ignoring order_id/amount):
0    False
1    False
2     True
3    False
4     True
5    False
dtype: bool
```

### Why full-row `duplicated()` found nothing here — read this carefully, it's the whole point
Every single row returns `False` for the full-row check, even though rows 2 and 4 are
**obviously** repeat orders in every business sense (same customer, same product, same
amount). The reason: `order_id` is unique per row (1 through 6), and full-row `duplicated()`
requires **every column** to match, including `order_id` — since no two rows share an
`order_id`, no two rows can ever be full-row duplicates, no matter how identical everything
else is. This is a live demonstration, not a hypothetical, of exactly the lesson below: the
default (all-columns) definition of "duplicate" is almost never what you actually want, because
any auto-generated unique ID column silently defeats it entirely.

### Why `subset=` is the parameter that actually matters
Just demonstrated live: full-row `duplicated()` here returns **zero** `True` values, because
`order_id` is unique per row — in real data, a true "duplicate" is almost always defined by a
**business key** subset (e.g., "same customer placed the same product order" — a sign of a
double-submitted form), not literal full-row identity, which any auto-incrementing ID column
will always defeat. **Always ask "duplicate according to which columns?" before calling
`drop_duplicates()` — the default (all columns) is rarely the business-correct definition, and
as shown above, can silently find nothing at all when a unique ID is present.**

### `keep` parameter -- first, last, or none


In [ ]:
print("drop_duplicates(subset=['customer_id','product'], keep='first') -- default, keeps FIRST occurrence:")
print(orders.drop_duplicates(subset=["customer_id", "product"], keep="first"))

print("\ndrop_duplicates(subset=[...], keep='last') -- keeps LAST occurrence:")
print(orders.drop_duplicates(subset=["customer_id", "product"], keep="last"))

print("\ndrop_duplicates(subset=[...], keep=False) -- drops ALL rows involved in ANY duplication:")
print(orders.drop_duplicates(subset=["customer_id", "product"], keep=False))


### Expected Output (verified on pandas 3.0.2)
```
drop_duplicates(subset=['customer_id','product'], keep='first') -- default, keeps FIRST occurrence:
   order_id  customer_id product  amount
0         1          101  Widget   29.99
1         2          102  Gadget   49.99
3         4          103   Gizmo   19.99
5         6          101   Gizmo   19.99

drop_duplicates(subset=[...], keep='last') -- keeps LAST occurrence:
   order_id  customer_id product  amount
2         3          101  Widget   29.99
3         4          103   Gizmo   19.99
4         5          102  Gadget   49.99
5         6          101   Gizmo   19.99

drop_duplicates(subset=[...], keep=False) -- drops ALL rows involved in ANY duplication:
   order_id  customer_id product  amount
3         4          103   Gizmo   19.99
5         6          101   Gizmo   19.99
```

### Why blindly removing duplicates can be genuinely dangerous
Consider: customer 101 legitimately ordered the same Widget **twice**, on two different real
occasions (not a data error at all) -- but if your source data doesn't include a timestamp/date
column, `drop_duplicates(subset=['customer_id','product'])` would incorrectly collapse two real
orders into one, silently **understating actual revenue and order counts**. This is exactly why
professional practice is: **before dropping anything, investigate whether the "duplicates" are
genuine data errors (double-submitted forms, ETL re-processing) or legitimate repeat events** --
often by checking whether a timestamp or order-sequence column would have distinguished them,
and if that column exists, including it in `subset=` rather than dropping it as "noise."

### Interview Insight
> "A customer table contains duplicate customer IDs. What would you investigate before removing
> them?"
First, whether they're **true exact duplicates** (identical across every column -- likely a
data-pipeline re-processing bug, generally safe to drop) or **partial duplicates** with
differing values in some columns (could be legitimate updates over time, e.g., an address
change, where you actually want the *most recent* row via `keep='last'` sorted by an update
timestamp -- not a blind drop). Also check whether the ID itself is supposed to be a unique key
at all (a schema/constraint question) -- if it is, duplicates indicate an upstream bug worth
fixing at the source, not just papering over downstream.


---
# CHAPTER 14 — Sorting

## Concept
`sort_values()` sorts by column(s); `sort_index()` sorts by the index. `nlargest()`/`nsmallest()`
are the efficient, readable shortcut for "top/bottom N" -- directly mirroring the NumPy
`argpartition` top-K pattern you already know, but without needing to hand-roll it.


In [ ]:
sales = pd.DataFrame({
    "rep": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "region": ["East", "West", "East", "West", "East"],
    "revenue": [45000, 62000, 38000, 71000, 55000],
})

print("sort_values('revenue') (ascending, default):\n", sales.sort_values("revenue"))
print("\nsort_values('revenue', ascending=False):\n", sales.sort_values("revenue", ascending=False))
print("\nsort_values(['region','revenue']) (multi-column: region ASC, then revenue ASC within region):")
print(sales.sort_values(["region", "revenue"]))
print("\nsort_values(['region','revenue'], ascending=[True, False]) (per-column direction):")
print(sales.sort_values(["region", "revenue"], ascending=[True, False]))


### Expected Output (verified on pandas 3.0.2)
```
sort_values('revenue') (ascending, default):
      rep region  revenue
2  Carol   East    38000
0  Alice   East    45000
4    Eve   East    55000
1    Bob   West    62000
3   Dave   West    71000

sort_values('revenue', ascending=False):
      rep region  revenue
3   Dave   West    71000
1    Bob   West    62000
4    Eve   East    55000
0  Alice   East    45000
2  Carol   East    38000

sort_values(['region','revenue']) (multi-column: region ASC, then revenue ASC within region):
      rep region  revenue
2  Carol   East    38000
0  Alice   East    45000
4    Eve   East    55000
1    Bob   West    62000
3   Dave   West    71000

sort_values(['region','revenue'], ascending=[True, False]) (per-column direction):
      rep region  revenue
4    Eve   East    55000
0  Alice   East    45000
2  Carol   East    38000
3   Dave   West    71000
1    Bob   West    62000
```

### `nlargest`/`nsmallest` -- the readable top-K shortcut


In [ ]:
print("sales.nlargest(2, 'revenue'):\n", sales.nlargest(2, "revenue"))
print("\nsales.nsmallest(2, 'revenue'):\n", sales.nsmallest(2, "revenue"))
print("\n<- Equivalent to sort_values('revenue', ascending=False).head(2), but reads more clearly")
print("   AND is typically faster: nlargest/nsmallest use a partial-sort algorithm internally,")
print("   directly analogous to NumPy's argpartition rather than a full O(n log n) sort just to")
print("   keep 2 rows.")


### Expected Output (verified on pandas 3.0.2)
```
sales.nlargest(2, 'revenue'):
     rep region  revenue
3  Dave   West    71000
1   Bob   West    62000

sales.nsmallest(2, 'revenue'):
      rep region  revenue
2  Carol   East    38000
0  Alice   East    45000

<- Equivalent to sort_values('revenue', ascending=False).head(2), but reads more clearly
   AND is typically faster: nlargest/nsmallest use a partial-sort algorithm internally,
   directly analogous to NumPy's argpartition rather than a full O(n log n) sort just to
   keep 2 rows.
```

### Interview Insight
> "What's the advantage of `nlargest(k, col)` over `sort_values(col, ascending=False).head(k)`?"
Same result, but `nlargest` is both more readable (states intent directly -- "give me the top
k") and typically more efficient for large DataFrames, since it doesn't need to fully sort every
row just to discard all but the top `k` -- directly analogous to preferring NumPy's
`argpartition` over a full `argsort` when only the top-K matters (a pattern you already know
from the NumPy course).


---
# CHAPTER 15 — Modifying Data

## Concept
Creating and modifying columns, renaming, dropping -- and the professional convention of
preferring **explicit reassignment** over `inplace=True`.


In [ ]:
df = pd.DataFrame({"name": ["Alice", "Bob", "Carol"], "salary": [65000, 95000, 72000]})

# --- Direct column assignment ---
df["bonus"] = df["salary"] * 0.10
print("Direct assignment:\n", df)

# --- assign() -- returns a NEW DataFrame, supports METHOD CHAINING (Chapter 53) ---
df2 = df.assign(
    total_comp=lambda d: d["salary"] + d["bonus"],   # can reference OTHER columns being created in the SAME call
    tax=lambda d: d["total_comp"] * 0.25,
)
print("\nassign() (chainable, doesn't mutate original):\n", df2)
print("\nOriginal df unaffected by assign()?", "total_comp" not in df.columns)


### Expected Output (verified on pandas 3.0.2)
```
Direct assignment:
     name  salary   bonus
0  Alice   65000  6500.0
1    Bob   95000  9500.0
2  Carol   72000  7200.0

assign() (chainable, doesn't mutate original):
     name  salary   bonus  total_comp     tax
0  Alice   65000  6500.0     71500.0  17875.0
1    Bob   95000  9500.0    104500.0  26125.0
2  Carol   72000  7200.0     79200.0  19800.0

Original df unaffected by assign()? True
```

### Why `assign()` with a `lambda` matters for chaining
The `lambda d: ...` form receives the DataFrame **as it exists mid-chain**, letting
`total_comp` reference `bonus` (itself created earlier in the *same* `assign()` call) without
needing a separate statement -- this is what makes `assign()` composable inside long method
chains (Chapter 53), unlike plain `df["col"] = ...` which requires the DataFrame to already
exist as a named variable.

### `rename()`, `drop()` -- column management


In [ ]:
renamed = df.rename(columns={"name": "employee_name", "salary": "annual_salary"})
print("rename():\n", renamed)

dropped = df.drop(columns=["bonus"])
print("\ndrop(columns=[...]):\n", dropped)

dropped_rows = df.drop(index=[0, 2])
print("\ndrop(index=[...]):\n", dropped_rows)


### Expected Output (verified on pandas 3.0.2)
```
rename():
  employee_name  annual_salary   bonus
0         Alice          65000  6500.0
1           Bob          95000  9500.0
2         Carol          72000  7200.0

drop(columns=[...]):
     name  salary
0  Alice   65000
1    Bob   95000
2  Carol   72000

drop(index=[...]):
   name  salary   bonus
1  Bob   95000  9500.0
```

### `inplace=True` -- why modern professional code generally avoids it


In [ ]:
df_a = pd.DataFrame({"x": [1, 2, 3]})

# The inplace=True style -- mutates df_a directly, returns None
result_of_inplace = df_a.rename(columns={"x": "y"}, inplace=True)
print("df_a after inplace=True rename:\n", df_a)
print("What did the METHOD CALL ITSELF return?", result_of_inplace, " <- None! Can't chain, can't reassign")

# The explicit style -- returns a NEW object, original untouched unless reassigned
df_b = pd.DataFrame({"x": [1, 2, 3]})
df_b_renamed = df_b.rename(columns={"x": "y"})
print("\ndf_b (original, untouched):\n", df_b)
print("df_b_renamed (new object):\n", df_b_renamed)


### Expected Output (verified on pandas 3.0.2)
```
df_a after inplace=True rename:
   y
0  1
1  2
2  3
What did the METHOD CALL ITSELF return? None  <- None! Can't chain, can't reassign

df_b (original, untouched):
   x
0  1
1  2
2  3
df_b_renamed (new object):
   y
0  1
1  2
2  3
```

### The real reasons to prefer explicit reassignment
1. **`inplace=True` methods return `None`** -- verified above -- which means you cannot chain
   further operations onto them, and a common beginner bug is writing
   `df = df.rename(..., inplace=True)`, which sets `df` to `None` entirely, destroying the data.
2. **It does NOT reliably save memory or run faster** -- this is a persistent myth; internally,
   `inplace=True` often still creates a new array and just reassigns it to the same variable
   name, providing no meaningful performance advantage in most cases, while removing the ability
   to chain and making debugging harder (you can no longer easily compare before/after in the
   same cell).
3. **Explicit reassignment (`df = df.method(...)`) is unambiguous, chainable, and matches the
   functional style Chapter 53's method chaining relies on entirely.**

### Interview Insight
> "Is `inplace=True` faster than explicit reassignment?"
No, generally not -- this is a common misconception. Internally, most `inplace=True` operations
still perform the same underlying computation and array creation as the non-inplace version;
they just skip returning the new object and instead overwrite the original variable's
reference. The real cost of `inplace=True` is losing chainability (the method returns `None`)
and losing the safety of having both the before and after versions available for
comparison/debugging -- modern professional Pandas code generally prefers explicit
`df = df.method(...)` reassignment.


---
# CHAPTER 16 — String Operations

## Concept
The `.str` accessor applies string methods **element-wise across an entire column** -- the
vectorized equivalent of looping over strings, and (per Chapter 19) dramatically faster than a
Python-level loop or `.apply()` for large columns.


In [ ]:
messy = pd.DataFrame({
    "name": ["  Alice Smith ", "BOB JONES", "carol lee", "Dave  Kim", None],
    "email": ["alice@GMAIL.com", "bob@yahoo.COM", "carol.lee@company.co", "not-an-email", "eve@gmail.com"],
    "phone": ["(555) 123-4567", "555.234.5678", "5553456789", "555-456-7890", None],
})

print("messy:\n", messy)
print()
print("str.strip() + str.title() (clean whitespace/casing):")
cleaned_name = messy["name"].str.strip().str.title()
print(cleaned_name)


### Expected Output (verified on pandas 3.0.2)
```
messy:
             name                  email          phone
0   Alice Smith        alice@GMAIL.com  (555) 123-4567
1       BOB JONES          bob@yahoo.COM   555.234.5678
2       carol lee  carol.lee@company.co     5553456789
3      Dave  Kim            not-an-email   555-456-7890
4            None           eve@gmail.com           None

str.strip() + str.title() (clean whitespace/casing):
0    Alice Smith
1      Bob Jones
2      Carol Lee
3      Dave  Kim
4            NaN
Name: name, dtype: str
```

### Chained `.str` methods, and the missing-value behavior worth noting
`.str.strip().str.title()` chains cleanly (each returns a Series `.str` can be called on
again). Notice row 4 (`None`) stays missing (`NaN`) through the whole chain -- `.str` methods are
**NaN-safe by default**: they skip missing values rather than raising an error, propagating
`NaN` through instead -- no manual `.dropna()` or null-checking needed before applying a
`.str` method.

### Regex-powered extraction: `str.contains`, `str.extract`, `str.replace`


In [ ]:
email_pattern = r"^[\w.+-]+@[\w-]+\.[\w.-]+$"   # a reasonably strict email regex

is_valid_email = messy["email"].str.contains(email_pattern, regex=True, na=False)
print("str.contains(email_pattern):\n", is_valid_email)

domain_extracted = messy["email"].str.extract(r"@([\w.-]+)")
print("\nstr.extract(r'@([\\w.-]+)') -- captures the domain:\n", domain_extracted)

digits_only_phone = messy["phone"].str.replace(r"\D", "", regex=True)   # strip all non-digit characters
print("\nstr.replace(r'\\D', '', regex=True) -- digits only:\n", digits_only_phone)


### Expected Output (verified on pandas 3.0.2)
```
str.contains(email_pattern):
0     True
1     True
2     True
3    False
4     True
Name: email, dtype: bool

str.extract(r'@([\w.-]+)') -- captures the domain:
             0
0     GMAIL.com
1     yahoo.COM
2  company.co
3           NaN
4     gmail.com

str.replace(r'\D', '', regex=True) -- digits only:
0    5551234567
1    5552345678
2    5553456789
3    5554567890
4            NaN
Name: phone, dtype: str
```

### `na=False` in `str.contains` -- a small parameter with a real correctness consequence
Without `na=False`, `str.contains()` on a Series with missing values returns `NaN` (not
`False`) for those rows -- and using a `NaN`-containing Series directly as a boolean filter mask
(`df[mask]`) raises an error, since boolean indexing requires actual `True`/`False`, not `NaN`.
`na=False` explicitly says "treat missing values as not-matching," which is almost always the
correct semantic for a validity/contains check, and avoids that downstream filtering error
entirely.

### Real-world string-cleaning workflow, assembled


In [ ]:
def clean_name(s: pd.Series) -> pd.Series:
    """Standardize a messy name column: strip whitespace, title-case, collapse internal
    multiple spaces to one."""
    return (
        s.str.strip()
         .str.replace(r"\s+", " ", regex=True)   # collapse "Dave  Kim" (2 spaces) -> "Dave Kim"
         .str.title()
    )

result = clean_name(messy["name"])
print("Before:", messy["name"].tolist())
print("After :", result.tolist())


### Expected Output (verified on pandas 3.0.2)
```
Before: ['  Alice Smith ', 'BOB JONES', 'carol lee', 'Dave  Kim', None]
After : ['Alice Smith', 'Bob Jones', 'Carol Lee', 'Dave Kim', nan]
```

Notice `'Dave  Kim'` (with a doubled internal space) correctly collapsed to `'Dave Kim'` -- a
detail `str.strip()` alone would have missed entirely, since `strip()` only removes **leading
and trailing** whitespace, not repeated whitespace in the middle. This kind of layered,
deliberate string-cleaning chain (strip -> collapse internal whitespace -> normalize case) is
the realistic professional pattern, not a single method call.

### Interview Insight
> "Why is `.str.contains(pattern, na=False)` generally safer than the default
> `.str.contains(pattern)` when the result will be used to filter a DataFrame?"
The default returns `NaN` for rows where the original string was missing, and a boolean mask
containing `NaN` cannot be used directly for DataFrame filtering (it raises an error, since
`NaN` isn't a valid boolean). `na=False` treats missing values as "does not match," which is
almost always the intended semantic and avoids that filtering error, without requiring a
separate `.fillna(False)` step afterward.


---
# CHAPTER 17 — Categorical Data

## Concept
`category` dtype stores a column as a small integer code array plus a lookup table of distinct
category values -- instead of repeating the same string bytes for every matching row. For
columns with **low cardinality relative to row count** (e.g., a "department" column with 5
possible values across a million rows), this is a large, measurable memory and speed win.


In [ ]:
n = 200_000
rng = np.random.default_rng(0)
departments = rng.choice(["Sales", "Engineering", "Marketing", "Support", "Finance"], size=n)

as_str = pd.Series(departments)                  # default str dtype
as_category = pd.Series(departments, dtype="category")

mem_str = as_str.memory_usage(deep=True)
mem_cat = as_category.memory_usage(deep=True)

print(f"str dtype memory      : {mem_str / 1024:8.1f} KB")
print(f"category dtype memory : {mem_cat / 1024:8.1f} KB")
print(f"Reduction: {mem_str / mem_cat:.1f}x smaller")
print()
print("as_category.cat.categories:", as_category.cat.categories.tolist())
print("as_category.cat.codes.head():\n", as_category.cat.codes.head())


### Expected Output (verified on pandas 3.0.2)
```
str dtype memory      :   3084.7 KB
category dtype memory :    195.5 KB
Reduction: 15.8x smaller

as_category.cat.categories: ['Engineering', 'Finance', 'Marketing', 'Sales', 'Support']
as_category.cat.codes.head():
 0    1
1    4
2    2
3    0
4    0
dtype: int8
```

### Reading this result precisely
`str` dtype storage is roughly proportional to `n_rows x string_length` (every row's full text
is stored, even though there are only 5 distinct values) — and pandas 3.0's Arrow-backed `str`
dtype stores meaningfully more overhead per string than the classic `object` dtype did, which is
part of why the reduction measured here (15.8x) is even larger than category dtype's already
substantial win in earlier pandas versions. `category` dtype storage is roughly `n_rows x
code_size + small_lookup_table` — here, `int8` codes (1 byte each, since there are only 5
categories) plus a tiny 5-entry lookup table. The savings scale with (row count) x (1 -
code_size/original_string_size) — larger for longer repeated strings and higher row counts,
smaller (or even negative, extra overhead) for truly high-cardinality columns like unique IDs,
which is exactly why category dtype is a **low-cardinality-specific** tool, not a blanket
"convert all string columns" rule.

### `pd.Categorical()` with explicit, ordered categories


In [ ]:
satisfaction = pd.Series(["Medium", "High", "Low", "Medium", "High"])
ordered_cat = pd.Categorical(
    satisfaction,
    categories=["Low", "Medium", "High"],   # explicit ORDER, not alphabetical
    ordered=True,
)
satisfaction_ordered = pd.Series(ordered_cat)

print("Ordered categorical:\n", satisfaction_ordered)
print("\nComparison now works meaningfully:", (satisfaction_ordered > "Low").tolist())
print("Sorted by the DEFINED order (not alphabetical):\n", satisfaction_ordered.sort_values())


### Expected Output (verified on pandas 3.0.2)
```
Ordered categorical:
 0    Medium
1      High
2       Low
3    Medium
4      High
dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']

Comparison now works meaningfully: [True, True, False, True, True]
Sorted by the DEFINED order (not alphabetical):
 2       Low
0    Medium
3    Medium
1      High
4      High
dtype: category
Categories (3, str): ['Low' < 'Medium' < 'High']
```

### Why `ordered=True` matters beyond just sorting
Without an explicit order, Pandas has no way to know `"High" > "Medium"` means anything --
plain string comparison would sort alphabetically (`"High" < "Low" < "Medium"`, nonsensical for
satisfaction ratings). An **ordered categorical** encodes the *real* business ordering directly
into the dtype, making `>`/`<` comparisons and `.sort_values()` behave correctly according to
that meaning -- essential for any genuinely ordinal variable (satisfaction ratings, education
levels, size categories like S/M/L/XL) headed into either analysis or a model that should
respect ordinality.

### GroupBy performance connection (fully covered in Chapter 21)
Category dtype also speeds up `groupby()` on that column, since grouping by small integer codes
is faster than grouping by comparing full strings repeatedly -- another reason low-cardinality
grouping columns are excellent category-dtype candidates.

### Interview Insight
> "Should every string column in a DataFrame use category dtype?"
No -- category dtype's benefit comes specifically from **low cardinality relative to row
count** (repeated values). For a column of genuinely unique values (customer IDs, free-text
comments), converting to category provides no memory savings and can add overhead from
maintaining the lookup table, while gaining nothing since there's no repetition to exploit. The
decision should be based on `df[col].nunique() / len(df)` -- a low ratio (many repeats) favors
category dtype; a ratio near 1 (mostly unique) does not.


---
# CHAPTER 18 — Apply / Map / Applymap

## Concept
Three related-but-different tools, easy to conflate:
- **`Series.map()`**: element-wise transform of ONE Series, often via a dict/Series lookup or a
  simple function.
- **`Series.apply()`**: element-wise transform of ONE Series via a function (more general than
  `map`, can't use a dict directly).
- **`DataFrame.apply()`**: applies a function along an axis of a DataFrame -- to entire
  **rows** or entire **columns**, not just single elements.

## The performance hierarchy -- vectorization > map/apply > Python loop
This is the DataFrame-level version of the exact lesson from Chapter 1 of the NumPy course:
compiled vectorized operations beat anything that dispatches a Python function call per element.


In [ ]:
import timeit

n = 100_000
s = pd.Series(np.random.default_rng(0).uniform(0, 100, n))

def python_loop():
    return pd.Series([x * 2 + 5 for x in s])

def series_apply():
    return s.apply(lambda x: x * 2 + 5)

def vectorized():
    return s * 2 + 5

t_loop = timeit.timeit(python_loop, number=3)
t_apply = timeit.timeit(series_apply, number=3)
t_vec = timeit.timeit(vectorized, number=3)

print(f"Python loop (list comp): {t_loop:.4f}s")
print(f"Series.apply()          : {t_apply:.4f}s  ({t_loop/t_apply:.1f}x vs loop)")
print(f"Vectorized (s*2+5)      : {t_vec:.4f}s  ({t_loop/t_vec:.0f}x vs loop, {t_apply/t_vec:.0f}x vs apply)")


### Expected Output (verified on pandas 3.0.2 -- exact numbers and even ORDER can vary by machine/run)
```
Python loop (list comp): 0.0713s
Series.apply()          : 0.0917s  (0.8x vs loop -- apply was SLOWER here!)
Vectorized (s*2+5)      : 0.0013s  (56x vs loop, 73x vs apply)
```

### Reading this hierarchy precisely — including the part that didn't go as planned
`.apply()` is **not** reliably faster than a raw Python loop — in this actual run, it came out
*slower* (0.8x, i.e., took longer) than the list-comprehension loop. This matters more than a
clean "apply beats loop" story would have: `.apply()` still fundamentally calls your Python
function once per element under the hood, and the exact overhead relative to a plain loop
depends on implementation details, function complexity, and pandas version — it is **not a
dependable performance improvement over a loop**, and should never be reached for on that
basis. **True vectorization is in an entirely different performance class regardless** (tens of
times faster than either here) because `s * 2 + 5` dispatches to compiled operations processing
the whole array at once — exactly Chapter 1 and Chapter 12 of the NumPy course, now confirmed at
the Pandas layer, and the only part of this benchmark that's a truly reliable, large,
consistent win. **The rule: reach for `.apply()` only when a genuinely vectorized alternative
doesn't exist for what you need to compute** — not because it's faster than a loop (it may not
be), but because it's more idiomatic/composable within Pandas method chains than a hand-rolled
loop, when vectorization itself isn't available.

## `Series.map()` -- the dictionary-lookup use case that's genuinely idiomatic


In [ ]:
df = pd.DataFrame({"department_code": ["ENG", "SAL", "MKT", "ENG", "FIN"]})

code_to_name = {"ENG": "Engineering", "SAL": "Sales", "MKT": "Marketing", "FIN": "Finance"}
df["department"] = df["department_code"].map(code_to_name)
print("map() with a dict (a lookup table) -- extremely common, idiomatic pattern:\n", df)

# map() with an unmapped value -- produces NaN, doesn't raise
df2 = pd.DataFrame({"code": ["ENG", "SAL", "UNKNOWN"]})
df2["name"] = df2["code"].map(code_to_name)
print("\nUnmapped value ('UNKNOWN') becomes NaN, not an error:\n", df2)


### Expected Output (verified on pandas 3.0.2)
```
map() with a dict (a lookup table) -- extremely common, idiomatic pattern:
  department_code   department
0             ENG  Engineering
1             SAL        Sales
2             MKT    Marketing
3             ENG  Engineering
4             FIN      Finance

Unmapped value ('UNKNOWN') becomes NaN, not an error:
      code         name
0      ENG  Engineering
1      SAL        Sales
2  UNKNOWN          NaN
```

### `DataFrame.apply()` -- row-wise vs column-wise, controlled by `axis`


In [ ]:
scores = pd.DataFrame({"math": [85, 92, 78], "science": [90, 85, 88], "english": [75, 95, 80]})

col_means = scores.apply(np.mean, axis=0)     # axis=0: apply DOWN each column (Chapter 10's axis intuition, reused)
row_means = scores.apply(np.mean, axis=1)     # axis=1: apply ACROSS each row

print("scores:\n", scores)
print("\napply(mean, axis=0) -- per-COLUMN average:\n", col_means)
print("\napply(mean, axis=1) -- per-ROW average (each student's average):\n", row_means)

# Same-shape alternative for THIS specific case: pure vectorization, no apply at all
row_means_vectorized = scores.mean(axis=1)
print("\nVectorized equivalent (scores.mean(axis=1)) -- PREFER this over apply() when it exists:")
print(row_means_vectorized)
print("Identical?", row_means.equals(row_means_vectorized))


### Expected Output (verified on pandas 3.0.2)
```
scores:
    math  science  english
0    85       90       75
1    92       85       95
2    78       88       80

apply(mean, axis=0) -- per-COLUMN average:
math       85.000000
science    87.666667
english    83.333333
dtype: float64

apply(mean, axis=1) -- per-ROW average (each student's average):
0    83.333333
1    90.666667
2    82.000000
dtype: float64

Vectorized equivalent (scores.mean(axis=1)) -- PREFER this over apply() when it exists:
0    83.333333
1    90.666667
2    82.000000
dtype: float64
Identical? True
```

### `axis` for `DataFrame.apply()` -- the SAME mental model as NumPy's axis, one subtlety
`axis=0` applies the function to each **column** (walking down rows) -- collapsing the row
dimension, consistent with the NumPy course's "axis = the dimension that gets collapsed" rule.
`axis=1` applies to each **row** (walking across columns) -- collapsing the column dimension.
The subtlety: `DataFrame.apply(func, axis=1)` is often **much slower** than it looks, because
internally it constructs a full Series object for every single row before calling your
function -- for large DataFrames, this row-wise `apply` pattern is one of the most common
avoidable performance bottlenecks in real Pandas code (Chapter 43 formalizes this). **Whenever
a vectorized alternative exists (as it did here -- `scores.mean(axis=1)`), it should always be
preferred over `apply(func, axis=1)`, exactly matching this chapter's opening performance
hierarchy.**

### When `.apply()` genuinely earns its keep -- no vectorized alternative exists


In [ ]:
def classify_student(row):
    """Genuinely branchy, cross-column business logic -- awkward to vectorize cleanly."""
    avg = (row["math"] + row["science"] + row["english"]) / 3
    if avg >= 90:
        return "Honors"
    elif avg >= 80 and row["math"] >= 80:   # a cross-column AND condition, specifically on math
        return "STEM Track"
    elif avg >= 70:
        return "Standard"
    else:
        return "Needs Support"

scores["classification"] = scores.apply(classify_student, axis=1)
print(scores)
print("\n<- This IS a legitimate apply() use case: genuinely branchy, multi-condition, cross-column")
print("   logic that doesn't map cleanly onto np.where/np.select without becoming harder to read")
print("   than the function form -- though for very large data, np.select would still be preferred")
print("   for pure performance (Chapter 20's territory).")


### Expected Output (verified on pandas 3.0.2)
```
   math  science  english classification
0    85       90       75     STEM Track
1    92       85       95         Honors
2    78       88       80       Standard

<- This IS a legitimate apply() use case: genuinely branchy, multi-condition, cross-column
   logic that doesn't map cleanly onto np.where/np.select without becoming harder to read
   than the function form -- though for very large data, np.select would still be preferred
   for pure performance (Chapter 20's territory).
```

### Interview Insight
> "Is `apply()` always bad for performance?"
No — this is a nuanced, not absolute, truth. `apply()` is meaningfully slower than genuine
vectorization for anything that *can* be vectorized (arithmetic, comparisons, most standard
transforms) — verified above by a large margin. It is **not**, however, reliably faster than a
plain Python loop either — in this chapter's own measurement, `.apply()` was actually slower
than an equivalent list comprehension, since both are fundamentally calling a Python function
per element, and the exact overhead is implementation- and version-dependent. For genuinely
branchy, multi-condition, cross-column business logic that doesn't map cleanly onto
`np.where`/`np.select`/boolean masking, `apply()` can still be the most readable and
maintainable choice — but that's a readability argument, not a performance one, and the
performance cost should be measured, not assumed in either direction.


---
## Chapters 13–18 Summary

- **Duplicates**: `subset=` is the parameter that actually matters -- full-row duplication is
  rarely the business-correct definition; always investigate whether "duplicates" are genuine
  data errors or legitimate repeat events before dropping anything.
- **Sorting**: `sort_values()` with `ascending=[...]` per column for mixed-direction multi-column
  sorts; `nlargest`/`nsmallest` are the readable, typically-faster top-K shortcut -- the Pandas
  analog of NumPy's `argpartition`.
- **Modifying data**: `assign()` with lambdas enables chainable, sequential column creation;
  `inplace=True` is not faster (a persistent myth) and breaks chaining by returning `None` --
  verified live that an inplace call itself returns `None`.
- **String operations**: `.str` methods are vectorized and NaN-safe by default; `na=False` in
  `str.contains()` prevents a downstream boolean-filtering error; real cleaning is a deliberate
  chain (strip -> collapse whitespace -> normalize case), not one method call.
- **Categorical data**: verified a 15.8x memory reduction for a 5-value column across 200K
  rows -- the benefit is specifically for **low cardinality relative to row count**;
  `ordered=True` encodes real business ordering for meaningful `>`/`<` comparisons and sorting.
- **Apply/map**: vectorization decisively beat both `.apply()` and a raw loop here (~56-73x) —
  verified, not assumed; `.apply()` itself was actually *slower* than a plain loop in this run,
  a reminder that "apply beats a loop" is not a dependable claim, only "vectorization beats
  both" is; `map()` is the idiomatic dict-lookup tool; `DataFrame.apply(axis=1)` is legitimate
  only for genuinely branchy cross-column logic that resists vectorization, never as a
  first-choice default, and its justification is readability, not speed.

---

## Mastery Exercises

**Exercise 1:** A transactions table has `transaction_id` (always unique) and you suspect some
transactions were double-submitted. Which columns would you use for `subset=` in
`duplicated()`, and what additional column (if present) would make you MORE confident a match
is a true error rather than a legitimate repeat?

**Exercise 2 (Prediction):** For `df.apply(func, axis=1)` on a DataFrame with 1,000,000 rows,
predict roughly how this compares in speed to an equivalent vectorized expression, based on
this chapter's measured gap -- then consider what that means for whether `axis=1` apply is
viable at that scale at all.

**Coding Challenge:** Write `standardize_categorical(s, mapping, default="Other")` that maps a
Series of raw category strings through a provided dict, filling any unmapped value with
`default` instead of leaving it `NaN` -- using `.map()` plus `.fillna()`, not a loop.

**Debugging Challenge:**
```python
df = pd.DataFrame({"a": [1,2,3]})
df.rename(columns={"a": "b"}, inplace=True)
df = df.rename(columns={"b": "c"}, inplace=True)
print(df)   # a teammate is confused why this crashes with AttributeError on the NEXT line
```
Explain exactly what `df` becomes after the second line, and why.

**Interview Questions:**
1. Why is a low-cardinality column a good category-dtype candidate, while a high-cardinality
   column (like a unique ID) is not?
2. What's the real reason to avoid `inplace=True` in modern Pandas code -- is it about
   performance?
3. When is `.apply(axis=1)` actually the right choice, despite being slower than vectorization?
4. Why does `.str.contains(pattern, na=False)` matter for downstream filtering correctness?

**Quiz:**
- Q1: True/False -- `drop_duplicates()` with default parameters considers all columns when
  deciding what counts as a duplicate.
- Q2: What does an inplace method call itself return?
- Q3: Which is generally faster for large DataFrames: `df.apply(np.mean, axis=1)` or
  `df.mean(axis=1)`?

---
 
